In [2]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

data = pd.read_csv('/kaggle/input/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv')

data['Attrition'] = data['Attrition'].map({'Yes': 1, 'No': 0})

data_encoded = pd.get_dummies(data, columns=['Department', 'JobRole', 'OverTime'], drop_first=True)
data_encoded = pd.get_dummies(data_encoded, drop_first=True)

X = data_encoded.drop('Attrition', axis=1)
y = data_encoded['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(solver='liblinear', max_iter=10000)
model.fit(X_train, y_train)

coefs = dict(zip(X.columns, model.coef_[0]))
print(f"OverTime_Yes Coefficient: {coefs.get('OverTime_Yes', 0):.4f}")
print(f"MonthlyIncome Coefficient: {coefs.get('MonthlyIncome', 0):.4f}")

y_pred_proba = model.predict_proba(X_test)
loss = log_loss(y_test, y_pred_proba)
print(f"Log-Loss: {loss:.4f}")

with open('attrition_model.pkl', 'wb') as file:
    pickle.dump(model, file)

with open('attrition_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

synthetic_input = X_test.iloc[[0]].copy()
syn_pred = loaded_model.predict_proba(synthetic_input)[:, 1]
print(f"\nPredicted Attrition Probability for Synthetic Input: {syn_pred[0]:.4f}")

OverTime_Yes Coefficient: 1.1251
MonthlyIncome Coefficient: -0.0001
Log-Loss: 0.3230

Predicted Attrition Probability for Synthetic Input: 0.0650
